# Efficiency measurements for Section 9

Measures the subword-only model (the paper's headline configuration) and the
published transformers on the same hardware, on GPU and on CPU:

- throughput (tweets per second, batch 32)
- batch-1 latency (one tweet at a time)
- single-thread CPU speed, for the "single core" claim
- model file size on disk, measured by saving it
- peak process RAM

The fastText file is never downloaded. The subword-only model is built without
an embedding table, so if this notebook completes, the model needs no external
file.

## How to run

1. Settings: Accelerator = GPU T4 x2, Internet = On
2. Leave `SMOKE = True` and Run All. About 5 minutes. It runs every step on a
   tiny scale, so a broken step fails here rather than half an hour in.
3. Set `SMOKE = False` and Run All again. About 45 minutes: ~14 for training,
   ~10 for downloading the transformer weights, the rest for timing.
4. Collect everything in `/kaggle/working/efficiency_results/`.

To close the tab during step 3: Save Version -> Save & Run All (Commit).

## Settings

In [1]:
SMOKE  = False            # True = 5-minute check of every step. False = the real run.
BRANCH = 'sabith'        # the code changes live here until merged into main
REPO   = 'https://github.com/hatheem-r/project_DNN.git'

if SMOKE:
    EPOCHS, LATENCY_OURS = 1, 20
    TRF_MODELS = ['--model', 'sinbert']          # smallest download, same code path
    GPU_LIMIT, CPU_LIMIT = 64, 16
    LATENCY_TRF_GPU, LATENCY_TRF_CPU = 3, 2
else:
    EPOCHS, LATENCY_OURS = 35, 500
    TRF_MODELS = ['--all']
    GPU_LIMIT, CPU_LIMIT = 0, 250                # 0 = all 2,500 tweets
    LATENCY_TRF_GPU, LATENCY_TRF_CPU = 100, 30   # XLM-R-large is ~1 s per tweet on CPU

print('SMOKE TEST' if SMOKE else 'FULL RUN', '| epochs', EPOCHS)

FULL RUN | epochs 35


## GPU

In [2]:
import torch, os
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('CPU cores available:', os.cpu_count())
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'

torch 2.10.0+cu128 | CUDA True
Tesla T4
CPU cores available: 4


## Code

In [3]:
import os, shutil, subprocess
os.chdir('/kaggle/working')
if os.path.exists('project'):
    shutil.rmtree('project')
!git clone -q -b $BRANCH $REPO project
os.chdir('/kaggle/working/project')
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
os.makedirs('results', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)
print('HEAD:', subprocess.run(['git', 'log', '-1', '--oneline'],
                              capture_output=True, text=True).stdout.strip())

gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
HEAD: 7b9565f Build the subword-only model without an embedding table; extend efficiency measurements


## Tests

The alignment tests check that piece lists line up with words. A break there
shifts labels against tokens without raising an error.

In [4]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2


all 9 tests passed
All alignment tests passed.



## Helper

Streams each command's output live and saves it to a text file. `gpu=False`
hides the GPU from that one command, so PyTorch runs it on the CPU while
reusing the model already trained on the GPU.

In [5]:
import os, subprocess

def run(args, logfile, gpu=True):
    env = dict(os.environ)
    if not gpu:
        env['CUDA_VISIBLE_DEVICES'] = ''
    print('$', ' '.join(args), '' if gpu else '   [GPU hidden]', flush=True)
    with open(logfile, 'w') as log:
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        rc = p.wait()
    print('exit code', rc)
    assert rc == 0, f'failed - see {logfile}'

## 1. Our model on GPU

Deletes any existing subword-only checkpoint and earlier efficiency results
first. Without that, a full run in the same session would silently reuse the
1-epoch model the smoke test trained. Then trains one seed (~14 minutes in the
full run) and measures it.

In [6]:
import glob, os
for f in (['artifacts/analysis_model_minimal.pt']
          + glob.glob('results/efficiency_minimal*')
          + glob.glob('results/transformer_efficiency*')):
    if os.path.exists(f):
        os.remove(f)
        print('removed', f)

run(['python', 'notebooks/10_analysis.py', '--efficiency', '--minimal',
     '--epochs', str(EPOCHS), '--latency-tweets', str(LATENCY_OURS)],
    'results/efficiency_minimal_gpu.txt')

log = open('results/efficiency_minimal_gpu.txt').read()
assert f'training one seed for {EPOCHS} epochs' in log, 'did not train a fresh model'
assert 'fastText vectors NOT loaded' in log, 'fastText was loaded'
assert 'word-embedding table   none' in log, 'model still has an embedding table'
print('checks passed: fresh model, no fastText, no embedding table')

$ python notebooks/10_analysis.py --efficiency --minimal --epochs 35 --latency-tweets 500 

0. SETUP




Generating train split: 100%|██████████| 7500/7500 [00:00<00:00, 33882.16 examples/s]

Generating test split: 100%|██████████| 2500/2500 [00:00<00:00, 38545.06 examples/s]
device cuda   train 7,500   test 2,500
checkpoint: artifacts/analysis_model_minimal.pt
training the tokenizer on the full train split
fastText vectors NOT loaded - the subword-only model has no word channel
model: subword only
tokenizer 1,000 pieces   vocab 33,006

EFFICIENCY MEASUREMENTS - FOR SECTION 9
no checkpoint; training one seed for 35 epochs (about 14 minutes on a T4)
  epoch 10/35  loss 1.5515
  epoch 20/35  loss 1.3464
  epoch 30/35  loss 1.2188
  epoch 35/35  loss 1.1674
saved artifacts/analysis_model_minimal.pt
parameters (measured):
  total                             176,058
  trainable                         176,058
  frozen                                  0
  word_embedding                      

## 2. Our model on CPU, all cores

Reuses the checkpoint from step 1. Compare this against the transformers in
step 5, which use the same thread setting.

In [7]:
run(['python', 'notebooks/10_analysis.py', '--efficiency', '--minimal',
     '--latency-tweets', str(LATENCY_OURS)],
    'results/efficiency_minimal_cpu.txt', gpu=False)

$ python notebooks/10_analysis.py --efficiency --minimal --latency-tweets 500    [GPU hidden]

0. SETUP
device cpu   train 7,500   test 2,500
checkpoint: artifacts/analysis_model_minimal.pt
fastText vectors NOT loaded - the subword-only model has no word channel
model: subword only
tokenizer 1,000 pieces   vocab 33,006

EFFICIENCY MEASUREMENTS - FOR SECTION 9
loaded artifacts/analysis_model_minimal.pt
parameters (measured):
  total                             176,058
  trainable                         176,058
  frozen                                  0
  word_embedding                          0
  subword_channel                    90,800
  sentence_head                           0
  lstm_input_dim                        100
  non_embedding_trainable           176,058

inference over the full test split, measured on cpu:
  tweets                 2,500
  real tokens            59,670
  wall clock             2.34s
  tweets per second      1,068
  tokens per second      25,490
  ms per 

## 3. Our model on one CPU thread

The paper says "a single CPU core". The earlier CPU figure used every core
available, so this is the measurement that claim needs.

In [8]:
run(['python', 'notebooks/10_analysis.py', '--efficiency', '--minimal',
     '--threads', '1', '--latency-tweets', str(LATENCY_OURS)],
    'results/efficiency_minimal_cpu_1thread.txt', gpu=False)

$ python notebooks/10_analysis.py --efficiency --minimal --threads 1 --latency-tweets 500    [GPU hidden]

0. SETUP
device cpu   train 7,500   test 2,500
checkpoint: artifacts/analysis_model_minimal.pt
fastText vectors NOT loaded - the subword-only model has no word channel
model: subword only
tokenizer 1,000 pieces   vocab 33,006

EFFICIENCY MEASUREMENTS - FOR SECTION 9
loaded artifacts/analysis_model_minimal.pt
parameters (measured):
  total                             176,058
  trainable                         176,058
  frozen                                  0
  word_embedding                          0
  subword_channel                    90,800
  sentence_head                           0
  lstm_input_dim                        100
  non_embedding_trainable           176,058

inference over the full test split, measured on cpu:
  tweets                 2,500
  real tokens            59,670
  wall clock             2.41s
  tweets per second      1,039
  tokens per second      24,7

## 4. Transformers on GPU

SinBERT, XLM-T, XLM-R-base and XLM-R-large with their public weights and an
untrained classification head. Inference only: nothing is fine-tuned, and no
accuracy is measured. Most of the time here is downloading about 5 GB.

In [9]:
args = ['python', 'notebooks/11_transformer_efficiency.py', *TRF_MODELS,
        '--latency-tweets', str(LATENCY_TRF_GPU)]
if GPU_LIMIT:
    args += ['--limit', str(GPU_LIMIT)]
run(args, 'results/transformer_efficiency_gpu.txt')
assert 'FAILED' not in open('results/transformer_efficiency_gpu.txt').read(), 'a model failed - see the log'

$ python notebooks/11_transformer_efficiency.py --all --latency-tweets 100 

SETUP
device: cuda   CPU threads used: 2 (of 4 available)
test tweets timed: 2,500

sinbert  (NLPC-UOM/SinBERT-large)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2642.63it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
RobertaForTokenClassification LOAD REPORT from: NLPC-UOM/SinBERT-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arc

## 5. Transformers on CPU

A sample of tweets rather than all 2,500: tweets per second is the comparable
figure, and the full split would take tens of minutes for XLM-R-large alone.

In [10]:
run(['python', 'notebooks/11_transformer_efficiency.py', *TRF_MODELS, '--allow-cpu',
     '--limit', str(CPU_LIMIT), '--latency-tweets', str(LATENCY_TRF_CPU)],
    'results/transformer_efficiency_cpu.txt', gpu=False)
assert 'FAILED' not in open('results/transformer_efficiency_cpu.txt').read(), 'a model failed - see the log'

$ python notebooks/11_transformer_efficiency.py --all --allow-cpu --limit 250 --latency-tweets 30    [GPU hidden]

SETUP
device: cpu   CPU threads used: 2 (of 4 available)
test tweets timed: 250

sinbert  (NLPC-UOM/SinBERT-large)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2273.29it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
RobertaForTokenClassification LOAD REPORT from: NLPC-UOM/SinBERT-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture;

## 6. One comparison table

Reads the JSON files the five steps wrote and puts every model side by side.
Same hardware, same thread setting within each column.

In [11]:
import glob, json, os, shutil

def load(pattern):
    hits = sorted(glob.glob(pattern))
    return json.load(open(hits[-1])) if hits else None

ours_gpu = load('results/efficiency_minimal_cuda_t*.json')
ours_cpu = [json.load(open(f)) for f in sorted(glob.glob('results/efficiency_minimal_cpu_t*.json'))]
ours_cpu_all = max(ours_cpu, key=lambda r: r['threads'])
ours_cpu_one = min(ours_cpu, key=lambda r: r['threads'])
trf_gpu = load('results/transformer_efficiency_cuda_t*.json') or []
trf_cpu = load('results/transformer_efficiency_cpu_t*.json') or []

rows = [('ours (subword only)', ours_gpu['parameters']['total'], ours_gpu['model_file_mb'],
         ours_gpu, ours_cpu_all)]
cpu_by_name = {r['model']: r for r in trf_cpu}
for g in trf_gpu:
    rows.append((g['model'], g['parameters'], g['size_mb'], g, cpu_by_name.get(g['model'])))

def tps(r): return f"{r['tweets_per_s']:,.0f}" if r else '-'
def b1(r):  return f"{r['latency_ms_median']:.2f}" if r and r.get('latency_ms_median') else '-'

print(f"CPU columns use {ours_cpu_all['threads']} threads for every model.")
print(f"{'model':<22}{'params':>14}{'MB':>9}{'GPU tw/s':>11}{'GPU b1 ms':>11}"
      f"{'CPU tw/s':>11}{'CPU b1 ms':>11}")
print('-' * 89)
for name, params, mb, g, c in rows:
    print(f"{name:<22}{params:>14,}{mb:>9.1f}{tps(g):>11}{b1(g):>11}{tps(c):>11}{b1(c):>11}")

print()
print(f"ours on ONE CPU thread: {ours_cpu_one['tweets_per_s']:,.0f} tweets/s, "
      f"batch-1 median {ours_cpu_one['latency_ms_median']:.2f} ms")
print(f"ours, file on disk: {ours_gpu['model_file_mb']:.2f} MB model + "
      f"{ours_gpu['tokenizer_file_mb']:.2f} MB tokenizer, no external files")
print(f"ours, peak process RAM on CPU: {ours_cpu_all['peak_process_ram_mb']} MB "
      '(whole Python process: libraries, data and model)')

print()
if SMOKE:
    print('SMOKE TEST PASSED. These numbers are meaningless (1 epoch, tiny samples).')
    print('Set SMOKE = False in the Settings cell and Run All again.')
else:
    out = '/kaggle/working/efficiency_results'
    os.makedirs(out, exist_ok=True)
    for f in glob.glob('results/efficiency_minimal*') + glob.glob('results/transformer_efficiency*'):
        shutil.copy(f, out)
    print('copied to', out + ':')
    for f in sorted(os.listdir(out)):
        print('  ', f)

CPU columns use 2 threads for every model.
model                         params       MB   GPU tw/s  GPU b1 ms   CPU tw/s  CPU b1 ms
-----------------------------------------------------------------------------------------
ours (subword only)          176,058      0.7      1,213       4.70      1,068       4.52
sinbert                  125,389,058    501.6        128       8.97          8     123.08
xlmt                     277,454,594   1109.8        142       9.71          9      82.44
xlmr-base                277,454,594   1109.8        145       8.27         11      74.40
xlmr-large               558,842,882   2235.4         47      18.32          3     303.44

ours on ONE CPU thread: 1,039 tweets/s, batch-1 median 3.97 ms
ours, file on disk: 0.71 MB model + 0.26 MB tokenizer, no external files
ours, peak process RAM on CPU: 851 MB (whole Python process: libraries, data and model)

copied to /kaggle/working/efficiency_results:
   efficiency_minimal_cpu.txt
   efficiency_minimal_cpu

## What to commit

Everything in `efficiency_results/` goes under `results/` in the repo:

```
git add results/efficiency_minimal* results/transformer_efficiency*
git commit -m "Section 9 efficiency measurements"
git pull --rebase
git push
```

For the paper, state the hardware (T4 / Kaggle CPU and its thread count), the
batch size, and which figures are measured versus cited.